# Eotvos correction

The Eotvos correction is a necessary correction to apply for any gravity surveys conducted on a moving platform, such as a ship or airplane. The correction accounts for the relative motion between the vehicle and the Earth's surface, which generates an additional centrifugal force known as the Eotvos effect. We offer several methods of computing the correction, which we show here, generally from simpliest to most complex.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import plotly.io as pio
import verde as vd

import airbornegeo

pio.renderers.default = "notebook"

## Load data

This is a subset of the BAS AGAP survey over Antarctica's Gamburtsev Subglacial Mountains. The file is downloaded and subset in the notebook `AGAP_gravity_survey`. It has a pre-computed Eotvos correction, which we will compare our computed values to.

In [ ]:
data_df = pd.read_csv("data/AGAP_gravity_survey.csv")
print(data_df.columns)
data_df.head()

In [ ]:
data_df["distance_along_line"] = airbornegeo.along_track_distance(
    data_df,
    groupby_column="line",
)

In [ ]:
# get only the raw columns
# we will perform the corrections ourselves and compare to their values
data_df = data_df[
    [
        "EotvosCor",
        "Lon",
        "Lat",
        "Height_WGS1984",
        "easting",
        "northing",
        "unixtime",
        "line",
        "distance_along_line",
    ]
]
data_df.head()

## Eotvos correction following Harlan 1968

Another common method to compute the correction is from [Harlan 1968](https://doi.org/10.1029/JB073i014p04675). The paper offers several equations, but first we will follow equation 15. 

### Equation 15:
This formulation requires the aircraft latitude, longtide, time (to calculate time derivatives) and height. 

```math
E = \frac{V_N^2}{a} [ 1 + \frac{h}{a} + f (2-3\sin^{2}\phi) ] + \frac{V_E^2}{a} [ 1 + \frac{h}{a} - f \sin^{2}\phi) ] + 2 V_E \omega \cos \phi (1 + \frac{h}{a})
```
where $V_N$, $V_E$, $h$, and $\phi$, $ are the aircraft's eastward and northward velocities, height, and geodetic latitude, and $a$, $f$, and $\omega$ are the ellipsoid's semimajor axis, flattening, and rotation rate.

```math
V_N = r' \dot{\phi_{c}} sec(D)
```
```math
V_E = r' \dot{l} cos(\phi_{c})
```

where $r'$ is the ellipsoid's geocentric radius at the latitude of the aircraft, $l$ is the aircraft's longitude, $\phi_{c}$ is the geocentric radius of the aircraft, $\dot{\phi_{c}}$ is the first time derivative of the geocentric radius of the aircraft, and $D$ is the deviation between the aircraft's geodetic latitude and geocentric latitude. 

Compared to the Glicken formulation, it is more exact since it accounts for aircraft height as well as the flattening of the ellipsoid. However it still makes a few assumptions which simplify the calculations. These include a simplification for the deviation between geocentric and geodetic latitudes, and thus the time derivatives of this deviation, as well as a simplification of the geocentric radius, and thus the time derivatives of the geocentric radius.

In [ ]:
data_df["eotvos_correction_harlan"] = airbornegeo.eotvos_correction_harlan(
    data_df.Lat.values,
    data_df.Lon.values,
    data_df.unixtime.values,
    data_df.Height_WGS1984.values,
)
data_df.head()

In [ ]:
ax = data_df.plot.scatter(
    "easting",
    "northing",
    c="eotvos_correction_harlan",
    s=0.2,
)
ax.set_aspect("equal")

In [ ]:
df = data_df[data_df.line == 4]
ylim = vd.minmax(df.EotvosCor)
ax = df.plot.line(
    "distance_along_line",
    "EotvosCor",
    style="bp",
    ms=0.6,
    label="Published values",
    title=f"Line {df.line.unique()[0]}",
    ylim=ylim,
)
ax = df.plot.line(
    "distance_along_line",
    "eotvos_correction_harlan",
    style="rp",
    ms=0.6,
    title=f"Line {df.line.unique()[0]}",
    label="Calculated values",
    ax=ax,
    ylim=ylim,
)
ax.set_ylabel("Eotvos correction (mGal)")

## Full Eotvos correction

The above methods used some assumptions to simplify the complex calculation for the Eotvos Correction. Below, we demonstrate the full analytic solution to the Eotvos correction equation. These methods utilize the 2nd temporal derivative of quantities such as latitude, longitude, and height, making them highly sensitive. If the data has been downsampled, the results will likely be skewed. 

In [ ]:
data_df["eotvos_correction_full"] = airbornegeo.eotvos_correction_full(
    data_df.Lat.values,
    data_df.Lon.values,
    data_df.unixtime.values,
    data_df.Height_WGS1984.values,
)
data_df.head()

In [ ]:
ax = data_df.plot.scatter(
    "easting",
    "northing",
    c="eotvos_correction_full",
    s=0.2,
)
ax.set_aspect("equal")

In [ ]:
df = data_df[data_df.line == 4]
# ylim = vd.minmax(df.EotvosCor)
ylim = vd.minmax(df.eotvos_correction_full)
ax = df.plot.line(
    "distance_along_line",
    "EotvosCor",
    style="bp",
    ms=0.6,
    label="Published values",
    title=f"Line {df.line.unique()[0]}",
    ylim=ylim,
)
ax = df.plot.line(
    "distance_along_line",
    "eotvos_correction_full",
    style="rp",
    ms=0.6,
    title=f"Line {df.line.unique()[0]}",
    label="Calculated values",
    ax=ax,
    ylim=ylim,
)
ax.set_ylabel("Eotvos correction (mGal)")

In [ ]:
data_df["eotvos_correction_approx"] = airbornegeo.eotvos_correction_approx(
    data_df.Lat.values,
    data_df.Lon.values,
    data_df.unixtime.values,
    data_df.Height_WGS1984.values,
)
data_df.head()

In [ ]:
ax = data_df.plot.scatter(
    "easting",
    "northing",
    c="eotvos_correction_approx",
    s=0.2,
)
ax.set_aspect("equal")

In [ ]:
df = data_df[data_df.line == 4]
# ylim = vd.minmax(df.EotvosCor)
ylim = vd.minmax(df.eotvos_correction_approx)
ax = df.plot.line(
    "distance_along_line",
    "EotvosCor",
    style="bp",
    ms=0.6,
    label="Published values",
    title=f"Line {df.line.unique()[0]}",
    ylim=ylim,
)
ax = df.plot.line(
    "distance_along_line",
    "eotvos_correction_approx",
    style="rp",
    ms=0.6,
    title=f"Line {df.line.unique()[0]}",
    label="Calculated values",
    ax=ax,
    ylim=ylim,
)
ax.set_ylabel("Eotvos correction (mGal)")

In [ ]:
df = data_df[data_df.line == 4]
df = df[df.distance_along_line < 140e3]
df = airbornegeo.resample(
    df,
    spacing=0.1,
    resample_by="unixtime",
    maxdist=10,
)
df.head()

In [ ]:
df.unixtime.iloc[0], df.unixtime.iloc[1]

In [ ]:
# from https://www.boulama.com/blog/posts/smoothing-noisy-sensor-data-with-kalman-filters.html
class KalmanFilter:
    def __init__(self, process_variance, measurement_variance):
        self.process_variance = process_variance
        self.measurement_variance = measurement_variance
        self.posteri_estimate = 0.0
        self.posteri_error_estimate = 1.0

    def update(self, measurement):
        priori_estimate = self.posteri_estimate
        priori_error_estimate = self.posteri_error_estimate + self.process_variance

        blending_factor = priori_error_estimate / (
            priori_error_estimate + self.measurement_variance
        )
        self.posteri_estimate = priori_estimate + blending_factor * (
            measurement - priori_estimate
        )
        self.posteri_error_estimate = (1 - blending_factor) * priori_error_estimate

        return self.posteri_estimate

In [ ]:
df["eotvos_correction_full"] = airbornegeo.eotvos_correction_full(
    df.Lat.values,
    df.Lon.values,
    df.unixtime.values,
    df.Height_WGS1984.values,
)

ylim = vd.minmax(
    df["eotvos_correction_full"],
    min_percentile=5,
    max_percentile=95,
)
ax = df.plot.line(
    "distance_along_line",
    "EotvosCor",
    style="rp",
    ms=1.6,
    ylim=ylim,
)
ax = df.plot.line(
    "distance_along_line",
    "eotvos_correction_full",
    style="bp",
    ms=0.4,
    ax=ax,
    ylim=ylim,
)

# df['eotvos_correction_full'] = sp.signal.savgol_filter(
#     x=df.eotvos_correction_full,
#     window_length=1000,
#     polyorder=2,
# )
# df['eotvos_correction_full'] = sp.signal.savgol_filter(
#     x=df.eotvos_correction_full,
#     window_length=1000,
#     polyorder=2,
# )
# df['eotvos_correction_full'] = sp.signal.savgol_filter(
#     x=df.eotvos_correction_full,
#     window_length=1000,
#     polyorder=2,
# )

# df['eotvos_correction_full'] = sp.signal.wiener(
#     df.eotvos_correction_full,
#     mysize=2001,
# )
# df['eotvos_correction_full'] = sp.signal.wiener(
#     df.eotvos_correction_full,
#     mysize=2001,
# )
# df['eotvos_correction_full'] = sp.signal.wiener(
#     df.eotvos_correction_full,
#     mysize=2001,
# )
# df['eotvos_correction_full'] = airbornegeo.filter_line(
#     df,
#     filter_type="g1000+l",
#     data_column='eotvos_correction_full',
#     filter_by_column="unixtime",
#     # pad_mode="linear_ramp",
#     pad_width_percentage=100,
# )
# df['eotvos_correction_full'] = airbornegeo.filter_line(
#     df,
#     filter_type="g1000+l",
#     data_column='eotvos_correction_full',
#     filter_by_column="unixtime",
#     # pad_mode="linear_ramp",
#     pad_width_percentage=100,
# )


kalman = KalmanFilter(
    process_variance=0.0000001,
    measurement_variance=10000,
)

# Apply Kalman filter
df["eotvos_correction_full"] = [
    kalman.update(measurement) for measurement in df.eotvos_correction_full
]


ylim = vd.minmax(
    df["eotvos_correction_full"],
    min_percentile=5,
    max_percentile=95,
)
ax = df.plot.line(
    "distance_along_line",
    "EotvosCor",
    style="rp",
    ms=1.6,
    ylim=ylim,
)
ax = df.plot.line(
    "distance_along_line",
    "eotvos_correction_full",
    style="bp",
    ms=0.4,
    ax=ax,
    ylim=ylim,
)
df.head()